In [1]:
# A. Menyisipkan Data ke MongoDB 
from pymongo import MongoClient
import random
import numpy as np

# 1. Koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["ecommerce_p13"]
koleksi = db["transaksi_mentah"]

# Bersihkan data lama (opsional, agar tidak duplikat jika di-run ulang)
koleksi.delete_many({})

# 2. Konfigurasi data dummy
random.seed(42)
np.random.seed(42)
n_dokumen = 800
kategori_list = ["Elektronik", "Fashion", "Makanan", "Kesehatan", "Olahraga"]
kota_list = ["Jakarta", "Bandung", "Surabaya", "Medan", "Makassar"]
metode_bayar_list = ["Transfer", "COD", "E-Wallet"]

dokumen_transaksi = []

for _ in range(n_dokumen):
    # Generate fitur dasar
    kategori = random.choice(kategori_list)
    kota = random.choice(kota_list)
    unit_terjual = np.random.randint(1, 10)
    harga_satuan = np.random.choice([50000, 100000, 250000, 500000])
    metode_bayar = random.choice(metode_bayar_list)
    estimasi_pengiriman = np.random.randint(1, 15) # 1-14 hari
    
    # Logika label: Semakin lama estimasi, semakin mungkin dibatalkan (probabilitas naik)
    # Menggunakan formula sigmoid sederhana
    threshold = 7
    prob_batal = 1 / (1 + np.exp(-(estimasi_pengiriman - threshold) * 0.8))
    label_dibatalkan = int(np.random.rand() < prob_batal)
    
    # Susun dokumen
    dokumen = {
        "kategori": kategori,
        "kota": kota,
        "unit_terjual": int(unit_terjual), # cast ke int murni
        "harga_satuan": int(harga_satuan),
        "metode_pembayaran": metode_bayar,
        "estimasi_pengiriman_hari": int(estimasi_pengiriman),
        "dibatalkan": label_dibatalkan
    }
    dokumen_transaksi.append(dokumen)

# 3. Insert ke MongoDB
koleksi.insert_many(dokumen_transaksi)
print(f"Berhasil menyisipkan {koleksi.count_documents({})} dokumen ke MongoDB.")

Berhasil menyisipkan 800 dokumen ke MongoDB.


In [2]:
# B. Ekstraksi ke Spark DataFrame
from pyspark.sql import SparkSession
import pandas as pd

# 1. Inisialisasi SparkSession (jika belum ada)
spark = SparkSession.builder \
    .appName("Tugas13_EcommerceRisk") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# 2. Tarik data dari MongoDB
dokumen_mongo = list(koleksi.find({}, {"_id": 0}))
df_pandas = pd.DataFrame(dokumen_mongo)

# 3. Konversi ke Spark DataFrame
df_transaksi = spark.createDataFrame(df_pandas)

# 4. Tampilkan Skema dan Jumlah Baris
print("Skema DataFrame:")
df_transaksi.printSchema()
print(f"Jumlah baris data: {df_transaksi.count()}")

26/11/26 14:27:43 WARN Utils: Your hostname, yan-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/26 14:27:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 14:27:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Skema DataFrame:
root
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- estimasi_pengiriman_hari: long (nullable = true)
 |-- dibatalkan: long (nullable = true)



[Stage 0:>                                                          (0 + 4) / 4]

Jumlah baris data: 800


In [3]:
# C. Melatih Model & Menyimpannya
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# 1. Definisikan Tahapan Pipeline
# StringIndexer untuk kolom kategorikal
idx_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx", handleInvalid="keep")
idx_metode = StringIndexer(inputCol="metode_pembayaran", outputCol="metode_idx", handleInvalid="keep")

# OneHotEncoder untuk kolom yang sudah di-index
ohe = OneHotEncoder(inputCols=["kategori_idx", "metode_idx"], 
                    outputCols=["kategori_vec", "metode_vec"])

# VectorAssembler untuk menggabungkan semua fitur numerik & vektor
assembler = VectorAssembler(
    inputCols=["kategori_vec", "metode_vec", "unit_terjual", "harga_satuan", "estimasi_pengiriman_hari"],
    outputCol="features"
)

# RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol="dibatalkan", numTrees=50, seed=99)

# Gabungkan ke dalam Pipeline
pipeline = Pipeline(stages=[idx_kategori, idx_metode, ohe, assembler, rf])

# 2. Split Data (80:20, seed=99)
train_data, test_data = df_transaksi.randomSplit([0.8, 0.2], seed=99)

# 3. Latih Model
model_pembatalan = pipeline.fit(train_data)
print("Model berhasil dilatih.")

# 4. Evaluasi Model
prediksi_test = model_pembatalan.transform(test_data)
evaluator_auc = BinaryClassificationEvaluator(labelCol="dibatalkan", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_score = evaluator_auc.evaluate(prediksi_test)

# Hitung Akurasi Manual
total_test = prediksi_test.count()
benar = prediksi_test.filter(prediksi_test.dibatalkan == prediksi_test.prediction).count()
akurasi = benar / total_test

print(f"Akurasi Model: {akurasi:.4f}")
print(f"AUC-ROC Score: {auc_score:.4f}")

# 5. Simpan Model
path_model = "model_pembatalan_tersimpan"
model_pembatalan.write().overwrite().save(path_model)
print(f"Model disimpan di folder: {path_model}")

Model berhasil dilatih.
Akurasi Model: 0.8663
AUC-ROC Score: 0.9258


Model disimpan di folder: model_pembatalan_tersimpan


In [4]:
# D. Simulasi "Minggu Berikutnya" 
from pyspark.ml import PipelineModel
import random

# 1. Hentikan SparkSession lama (Simulasi Jupyter ditutup)
spark.stop()
print("SparkSession lama ditutup.")

# 2. Buat SparkSession BARU
spark = SparkSession.builder \
    .appName("Tugas13_SesiBaru") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 3. Muat Model TANPA melatih ulang
model_dimuat = PipelineModel.load(path_model)

# 4. Buat 10 Data Transaksi Baru
data_baru_list = []
for _ in range(10):
    data_baru_list.append({
        "kategori": random.choice(kategori_list),
        "kota": random.choice(kota_list),
        "unit_terjual": random.randint(1, 10),
        "harga_satuan": random.choice([50000, 100000, 250000, 500000]),
        "metode_pembayaran": random.choice(metode_bayar_list),
        "estimasi_pengiriman_hari": random.randint(1, 14),
        "dibatalkan": 0 # Nilai dummy (tidak digunakan untuk prediksi)
    })

df_baru = spark.createDataFrame(pd.DataFrame(data_baru_list))

# 5. Lakukan Prediksi
hasil_prediksi = model_dimuat.transform(df_baru)
print("Hasil Prediksi 10 Transaksi Baru (1.0 = Berisiko Batal, 0.0 = Aman):")
hasil_prediksi.select("kategori", "estimasi_pengiriman_hari", "metode_pembayaran", "prediction", "probability").show(truncate=False)

SparkSession lama ditutup.


Hasil Prediksi 10 Transaksi Baru (1.0 = Berisiko Batal, 0.0 = Aman):
+----------+------------------------+-----------------+----------+----------------------------------------+
|kategori  |estimasi_pengiriman_hari|metode_pembayaran|prediction|probability                             |
+----------+------------------------+-----------------+----------+----------------------------------------+
|Kesehatan |5                       |COD              |0.0       |[0.8721556964887414,0.12784430351125856]|
|Elektronik|12                      |E-Wallet         |1.0       |[0.07214524792029553,0.9278547520797045]|
|Elektronik|14                      |Transfer         |1.0       |[0.10539875217461009,0.8946012478253899]|
|Kesehatan |1                       |Transfer         |0.0       |[0.8116166956954127,0.18838330430458727]|
|Makanan   |5                       |COD              |0.0       |[0.8254913846074278,0.17450861539257223]|
|Kesehatan |14                      |Transfer         |1.0       |[

In [6]:
# E. Menyimpan Laporan Akhir
# Simpan hasil prediksi sebagai Parquet, dipartisi berdasarkan 'kategori'
path_laporan = "laporan_risiko_pembatalan"
hasil_prediksi.write.mode("overwrite").partitionBy("kategori").parquet(path_laporan)
print(f"Laporan berhasil disimpan ke folder: {path_laporan}")

# Verifikasi dengan membaca kembali
df_verifikasi = spark.read.parquet(path_laporan)
print(f"Jumlah data yang berhasil dibaca kembali dari Parquet: {df_verifikasi.count()}")

# Tampilkan struktur folder partisi (opsional, untuk memastikan partisi terbentuk)
print("\nVerifikasi Partisi (Struktur Folder):")
import os
for root, dirs, files in os.walk(path_laporan):
    if dirs:
        print(f"Folder Partisi Ditemukan: {dirs}")
        break

Laporan berhasil disimpan ke folder: laporan_risiko_pembatalan
Jumlah data yang berhasil dibaca kembali dari Parquet: 10

Verifikasi Partisi (Struktur Folder):
Folder Partisi Ditemukan: ['kategori=Makanan', 'kategori=Elektronik', 'kategori=Olahraga', 'kategori=Kesehatan', 'kategori=Fashion']


Kesimpulan dan Alur Kerja End-to-End:
Alur kerja analitik Big Data yang telah dibangun dimulai dari tahap pengumpulan data (data ingestion) dengan menyimulasikan sistem kasir e-commerce yang menyimpan 800 data transaksi mentah ke dalam MongoDB berkat fleksibilitas skemanya. Data tersebut kemudian diekstraksi dan dikonversi menjadi Spark DataFrame untuk memanfaatkan kemampuan distributed processing dalam mengolah data berskala besar. Pada tahap pemodelan, dibangun sebuah Pipeline Machine Learning yang mengintegrasikan StringIndexer, OneHotEncoder, VectorAssembler, dan algoritma RandomForestClassifier untuk memprediksi risiko pembatalan pesanan berdasarkan fitur kategori, kota, dan estimasi pengiriman, yang kemudian dievaluasi menggunakan metrik Akurasi dan AUC-ROC. Melalui prinsip Model Persistence, model yang telah dilatih disimpan ke dalam disk dan berhasil dimuat kembali pada sesi SparkSession yang benar-benar baru tanpa perlu melakukan proses pelatihan ulang (fit), membuktikan efisiensi sistem dalam menyimulasikan prediksi mingguan. Terakhir, hasil prediksi disimpan dalam format Parquet yang terpartisi berdasarkan kategori, memanfaatkan columnar storage untuk memastikan pembacaan data yang jauh lebih cepat dan efisien untuk kebutuhan pelaporan bisnis.

Rekomendasi Bisnis (Actionable Insight):
Berdasarkan temuan model yang menunjukkan bahwa fitur estimasi_pengiriman_hari memiliki pengaruh logis yang sangat besar terhadap probabilitas pembatalan (semakin lama estimasi, semakin tinggi risiko batal), manajemen operasional disarankan untuk memprioritaskan optimalisasi logistik pada wilayah atau kategori produk tertentu yang memiliki estimasi pengiriman di atas rata-rata (misalnya di atas 7 hari). Selain itu, perusahaan dapat menerapkan strategi intervensi proaktif seperti memberikan voucher kompensasi atau notifikasi real-time kepada pelanggan yang pesanannya diprediksi memiliki risiko pembatalan tinggi (prediction = 1.0) sebelum pelanggan tersebut benar-benar membatalkan pesanan, sehingga dapat menekan angka cancellation rate secara signifikan.

In [7]:
# Penutup & Pembersihan
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup dengan bersih.")

Koneksi MongoDB & SparkSession ditutup dengan bersih.
